In [ ]:
import numpy as np
import pandas as pd

# 1.	Load the Dataset

In [ ]:
data = pd.read_csv("dirty_cafe_sales.csv")

In [ ]:
data.head()

In [ ]:
data.tail()

In [ ]:
data.info()

In [ ]:
data.describe()

In [ ]:
# Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
## Rename columns
data.columns = ["Transaction_ID", "Item", "Quantity", "Price_Per_Unit", "Total_Spent", "Payment_Method", "Location", "Transaction_Date"]
data.head()


In [ ]:
data.info()

# 2.	Data Type Correction

In [ ]:
#Numeric columns should be converted to numeric data types

data["Quantity"] = pd.to_numeric(data["Quantity"], errors="coerce")
data["Price_Per_Unit"] = pd.to_numeric(data["Price_Per_Unit"], errors="coerce")
data["Total_Spent"] = pd.to_numeric(data["Total_Spent"], errors="coerce")

# Date column should be converted to datetime data type
data["Transaction_Date"] = pd.to_datetime(data["Transaction_Date"], errors="coerce")

In [ ]:
data.info()

# 3.	Handle Missing and Invalid Values

In [ ]:

data.replace(["ERROR", "UNKNOWN", ""], np.nan, inplace=True)

In [ ]:
data.head()

In [ ]:
data.info()

In [ ]:
data.isnull().sum()

## Numerical columns

In [ ]:
# Fill Price_Per_Unit missing values by dividing Total_Spent by Quantity

data["Price_Per_Unit"] = data["Price_Per_Unit"].fillna(
    data["Total_Spent"] / data["Quantity"]
    )

In [ ]:
# Fill Quantity missing values by dividing Total_Spent by Price_Per_Unit
data["Quantity"] = data["Quantity"].fillna(
    data["Total_Spent"] / data["Price_Per_Unit"]
    )

In [ ]:
data.isnull().sum()

In [ ]:
# Fill non values in Quantity with median
data["Quantity"] = data["Quantity"].fillna(
    data["Quantity"].median()
)

In [ ]:
data.isnull().sum()

In [ ]:
# Fill non values in Price_Per_Unit with median
data["Price_Per_Unit"] = data["Price_Per_Unit"].fillna(
    data["Price_Per_Unit"].median()
)

In [ ]:
data.isnull().sum()

In [ ]:
# Fill missing values in Total_Spent by calculating Quantity * Price_Per_Unit
data["Total_Spent"] = data["Total_Spent"].fillna(
    data["Quantity"] * data["Price_Per_Unit"]
    )

In [ ]:
data.isnull().sum()

## Categorical columns

In [ ]:
# Fill missing values in Payment_Method with the most common payment method
most_common_payment_method = data["Payment_Method"].mode()[0]
data["Payment_Method"] = data["Payment_Method"].fillna(most_common_payment_method)

In [ ]:
data.isnull().sum()

In [ ]:
# Fill missing values in Location with the most common location
most_common_location = data["Location"].mode()[0]
data["Location"] = data["Location"].fillna(most_common_location)

In [ ]:
data.isnull().sum()

# 4.	Data Consistency and Integrity

In [ ]:
# 
df = data.iloc[[0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19]]
df

In [ ]:
# Fill missing Items according to price_per_unit
price_item_map = (
    data.groupby("Price_Per_Unit")["Item"]
        .agg(lambda s: s.mode().iat[0])   
)

data["Item"] = data["Item"].fillna(
    data["Price_Per_Unit"].map(price_item_map)
)

In [ ]:
data.isnull().sum()

In [ ]:
df = data.iloc[[0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19]]
df

In [ ]:
data.dropna(subset=["Transaction_Date"], inplace=True)

In [ ]:
data.isnull().sum()

In [ ]:
#Create a new column "Season" based on the month of the transaction date
def get_season(date):
    month = date.month
    
    if month in [12, 1, 2]:
        return "Winter"
    elif month in [3, 4, 5]:
        return "Spring"
    elif month in [6, 7, 8]:
        return "Summer"
    else:
        return "Fall"

data["Season"] = data["Transaction_Date"].apply(get_season)

In [ ]:
data.head()

In [ ]:
data

In [ ]:
data.info()

In [ ]:
# print dupplicate rows
df = data.duplicated().sum()
print(f"Number of duplicate rows: {df}")

In [ ]:
# Save the cleaned data to a new CSV file
data.to_csv("cleaned_cafe_sales.csv", index=False)